In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Load the dataset
file_path = "/content/drive/MyDrive/nosdra_2026-08-23_15_32_02UTC_complete.csv"
df = pd.read_csv(file_path, on_bad_lines='skip')

print(f"Original Dataset Shape: {df.shape}")

# 2. Re-engineer the Target Variable (Severity)
df['estimatedquantity'] = pd.to_numeric(df['estimatedquantity'], errors='coerce')

def categorize_severity(row):
    qty = row['estimatedquantity']
    habitat = str(row.get('spillareahabitat', '')).lower()

    if pd.isna(qty): return 'Unknown'
    if 'inland' in habitat:
        if qty > 250: return 'Major'
        elif qty >= 25: return 'Medium'
        else: return 'Minor'
    else:
        if qty > 2500: return 'Major'
        elif qty >= 250: return 'Medium'
        else: return 'Minor'

df['severity'] = df.apply(categorize_severity, axis=1)

# Drop rows where severity is 'Unknown' (we cannot train a supervised model without labels)
df_clean = df[df['severity'] != 'Unknown'].copy()
print(f"Shape after dropping 'Unknown' severity labels: {df_clean.shape}")

# 3. Drop Excluded Columns (Identified in Phase 2 logic)
cols_to_drop = [
    'id', 'incidentnumber', 'descriptionofimpact', 'attachments',
    'updatefor', 'zonaloffice', 'reportdate', 'estimatedquantity',
    'quantityrecovered', 'spillstopdate', 'typeoffacility',
    'initialcontainmentmeasures', 'latitude', 'longitude', 'lga',
    'estimatedspillarea', 'formadate', 'formbdate', 'formcdate',
    'jivdate', 'jivpresent', 'cleanupdate', 'cleanupcompleteddate',
    'cleanupmethods', 'postcleanupinspectiondate', 'postimpactassessmentdate',
    'remediationstart', 'remediationend', 'remediationtype', 'finalsamplingdate',
    'finallabresultsdate', 'certificatedate', 'certificatenumber', 'lastupdatedby'
]
# Use intersection to avoid errors if some columns are already dropped or missing
df_clean.drop(columns=df_clean.columns.intersection(cols_to_drop), inplace=True)

# 4. Feature Engineering: Extract temporal data
df_clean['incidentdate'] = pd.to_datetime(df_clean['incidentdate'], errors='coerce')
df_clean['incident_year'] = df_clean['incidentdate'].dt.year
df_clean['incident_month'] = df_clean['incidentdate'].dt.month
# Drop the original date string column as tree algorithms cannot process datetimes
df_clean.drop(columns=['incidentdate'], inplace=True)

# 5. Missing Value Imputation (Business Logic Applied)
# Categorical imputation (placeholder text)
cat_cols = df_clean.select_dtypes(include=['object']).columns
df_clean[cat_cols] = df_clean[cat_cols].fillna('unknown')

# Numerical imputation (median)
num_cols = df_clean.select_dtypes(include=['number']).columns
for col in num_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print(f"Final Preprocessed Shape: {df_clean.shape}")
print("Missing values successfully imputed and temporal features engineered.")

Original Dataset Shape: (21107, 42)
Shape after dropping 'Unknown' severity labels: (13202, 43)
Final Preprocessed Shape: (13202, 10)
Missing values successfully imputed and temporal features engineered.


In [3]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import joblib
import os

# 1. Label Encoding
# Identify all categorical columns
cat_cols = df_clean.select_dtypes(include=['object']).columns

# Dictionary to store the serialized encoders
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    # Fit and transform the data
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    encoders[col] = le

# Serialize (save) the encoders for future deployment or Phase 6 Explainability
os.makedirs('models', exist_ok=True)
joblib.dump(encoders, 'models/label_encoders.pkl')
print("Categorical variables successfully encoded and serialized.")

# 2. Data Splitting (80/20 Stratified)
# Separate features (X) and target variable (y)
X = df_clean.drop(columns=['severity'])
y = df_clean['severity']

# Perform the stratified split with a fixed random state
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

# 3. Save the prepared datasets for Phase 5 (Exclude from version control later)
os.makedirs('data/processed', exist_ok=True)
X_train.to_csv('data/processed/X_train.csv', index=False)
X_test.to_csv('data/processed/X_test.csv', index=False)
y_train.to_csv('data/processed/y_train.csv', index=False)
y_test.to_csv('data/processed/y_test.csv', index=False)

print("Processed datasets saved successfully.")

Categorical variables successfully encoded and serialized.
Training features shape: (10561, 9)
Testing features shape: (2641, 9)
Processed datasets saved successfully.


In [4]:
# 3. Save the prepared datasets for Phase 5 to Google Drive
save_dir = '/content/drive/MyDrive/processed_data'
os.makedirs(save_dir, exist_ok=True)

X_train.to_csv(f'{save_dir}/X_train.csv', index=False)
X_test.to_csv(f'{save_dir}/X_test.csv', index=False)
y_train.to_csv(f'{save_dir}/y_train.csv', index=False)
y_test.to_csv(f'{save_dir}/y_test.csv', index=False)

print(f"Processed datasets saved successfully to {save_dir}.")

Processed datasets saved successfully to /content/drive/MyDrive/processed_data.
